### <font color="greenblue"> Ponderação rake: Raking of replicate weight design - Iterative Proportional Fitting (IPF) </font>

| id | sexo | idade | peso (inicial) |
| -: | :--: | :---: | :------------: |
|  1 |   M  |   J   |        1       |
|  2 |   M  |   J   |        1       |
|  3 |   M  |   J   |        1       |
|  4 |   M  |   A   |        1       |
|  5 |   M  |   A   |        1       |
|  6 |   M  |   A   |        1       |
|  7 |   M  |   A   |        1       |
|  8 |   F  |   J   |        1       |
|  9 |   F  |   A   |        1       |
| 10 |   F  |   A   |        1       |

Contagens na amostra:  
- Sexo: M=7, F=3  
- Idade: J=4, A=6  

**Margens populacionais (alvos)**  
Vamos supor que na população:  
- Sexo: M=50%, F=50% de um total “10.000”  
então alvo: M=5.000, F=5.000  

- Idade: J=30%, A=70%  
então alvo: J=3.000, A=7.000  

Repare: o total geral é 10.000 em ambos os casos.

### Iteração 1 - Ajuste por SEXO

Como os pesos estão todos 1, o total ponderado atual por sexo é só a contagem:
- T_amostra_M = 7  
- T_amostra_F = 3  

Fatores:
- rM = 5000/7 = 714,2857
- rF = 5000/3 = 1666,6667

Agora multiplica:
- todo **homem** passa a ter peso **714,2857**
- toda **mulher** passa a ter peso **1666,6667**

Checagem (bate o sexo):

- Soma pesos de homens: 7×714,2857 = 5000
- Soma pesos de mulheres: 3×1666,6667 = 5000

✅ Sexo calibrado.

### Iteração 2 - Ajuste por IDADE (usando os pesos já alterados)
Agora calcule os totais ponderados por idade:

Quem é J? ids 1,2,3 (M,J) e id 8 (F,J)
- 3 homens jovens com peso 714,2857 → 3×714,2857 = 2142,8571  
- 1 mulher jovem com peso 1666,6667 → 1x1666,6667 = 1666,6667
- T_chapeu_J = 2142,8571 + 1666,6667 = 3809,5238
- T_chapeu_A = 4x714,2857 + 2x1666,6667 = 6190,4762

**Fatores da Idade:**
- rJ = 3000/3809,5238 ≈ 0,7875
- rA​ = 7000/6190,4762 ≈ 1,1310

Multiplica:
- todos os **J** têm seu peso multiplicado por **0,7875**
- todos os **A** multiplicado por **1,1310**

✅ Idade calibrada.

⚠️ Mas agora o sexo “descalibra” um pouco, porque parte dos homens/mulheres está em J e parte em A e receberam multiplicadores diferentes.

Iteração - 2 (volta para o SEXO)

In [42]:
import numpy as np
import pandas as pd

def rake_weights(df, weight_col, margins, control=None):
    """
    Raking (IPF) por margens 1D.
    
    df: DataFrame com colunas das variáveis de margem
    weight_col: nome da coluna de peso a ajustar
    margins: dict {var: {categoria: total_pop}}
             ex.: {"sexo": {"M": 5200, "F": 4800}, "idade": {"18-34": 3000, ...}}
    control: dict com "maxit" e "epsilon"
    """
    if control is None:
        control = {"maxit": 100, "epsilon": 1e-10}

    w = df[weight_col].to_numpy(dtype=float)

    # Checagens básicas (missing em variáveis de margem quebra o algoritmo)
    for var in margins.keys():
        if df[var].isna().any():
            raise ValueError(f"Há missing na variável de margem '{var}'. Trate antes de rakear.")

    for it in range(control["maxit"]):
        w_old = w.copy()
        max_rel_change = 0.0

        # Ajusta uma margem por vez
        for var, pop_totals in margins.items():
            # Totais atuais por categoria (ponderados)
            current = (
                pd.DataFrame({var: df[var].values, "_w": w})
                .groupby(var)["_w"].sum()
                .to_dict()
            )

            # Aplica fator categoria-a-categoria
            for cat, pop_total in pop_totals.items():
                cur_total = current.get(cat, 0.0)

                # Se a categoria não aparece na amostra, não dá para ajustar (divisão por zero)
                if cur_total <= 0:
                    raise ValueError(
                        f"Categoria '{cat}' (var='{var}') tem total atual 0 na amostra. "
                        f"Rake não consegue ajustar: revise categorias/margens/amostra."
                    )

                ratio = pop_total / cur_total
                mask = (df[var].values == cat)
                w[mask] *= ratio

        # Critério de convergência: maior mudança relativa de peso
        rel_change = np.max(np.abs(w - w_old) / (np.abs(w_old) + 1e-12))
        max_rel_change = max(max_rel_change, rel_change)

        if max_rel_change < control["epsilon"]:
            break

    return w

def make_bootstrap_replicates(df, base_weight_col, R=30, seed=123):
    """
    Cria replicate weights por bootstrap simples:
    - reamostra índices com reposição
    - replicate weight = base_weight * contagem_de_vezes_que_o_i_aparece
    """
    rng = np.random.default_rng(seed)
    n = len(df)
    base_w = df[base_weight_col].to_numpy(dtype=float)

    Wrep = np.zeros((n, R), dtype=float)
    for r in range(R):
        sample_idx = rng.integers(0, n, size=n)  # n draws with replacement
        counts = np.bincount(sample_idx, minlength=n)
        Wrep[:, r] = base_w * counts
    return Wrep

def check_margins(df, w, margins):
    """
    Retorna um dict com totais ponderados por categoria, para checar se bate com a população.
    """
    out = {}
    for var, pop_totals in margins.items():
        tot = (
            pd.DataFrame({var: df[var].values, "_w": w})
            .groupby(var)["_w"].sum()
            .reindex(pop_totals.keys())
        )
        out[var] = pd.DataFrame({
            "weighted_total": tot.values,
            "population_total": [pop_totals[k] for k in pop_totals.keys()],
        }, index=list(pop_totals.keys()))
    return out

In [43]:
# -----------------------------
# EXEMPLO TOY (você pode trocar pelos seus dados)
# -----------------------------
df = pd.DataFrame({
    "sexo": ["M","F","F","M","F","M","M","F","M","F"],
    "idade": ["18-34","18-34","35-54","35-54","55+","55+","18-34","35-54","55+","18-34"],
})

# Peso inicial (ex.: todos 1 só para simplificar)
df["w_base"] = 1.0
df

,sexo,idade,w_base
0,M,18-34,1.0
1,F,18-34,1.0
2,F,35-54,1.0
3,M,35-54,1.0
4,F,55+,1.0
5,M,55+,1.0
6,M,18-34,1.0
7,F,35-54,1.0
8,M,55+,1.0
9,F,18-34,1.0


In [44]:
path = r"C:\Users\rayner.santos\Downloads\Consistencia e Processamento\Cielo\EMP_Cielo Satisfacao_3onda_NOV25_2025.12.26.xlsx"
data = pd.read_excel(path, sheet_name="BD_CODIGOS")

In [45]:
df = data[["ID_EMP", "ONDA", "TRAB_C", "PF_PJ", "SEG_NOVO_BU", "VREG", "POND"]].copy()
df = df.loc[(df["TRAB_C"] == 1) & (df["ONDA"] == 59)]
df["w_base"] = 1.0
df

,ID_EMP,ONDA,TRAB_C,PF_PJ,SEG_NOVO_BU,VREG,POND,w_base
0,3000230,59,1,2,2,2,0.999119,1.0
2,3000383,59,1,2,1,2,0.317763,1.0
9,3001301,59,1,1,1,2,0.470412,1.0
10,3000014,59,1,2,2,3,0.994190,1.0
13,3000716,59,1,2,2,1,1.340074,1.0
...,...,...,...,...,...,...,...,...
15351,3004051,59,1,2,3,3,1.167923,1.0
15352,3004052,59,1,2,3,4,0.811759,1.0
15353,3004053,59,1,2,3,2,1.183012,1.0
15354,3004054,59,1,2,3,3,1.167923,1.0


In [46]:
for col in ["PF_PJ", "SEG_NOVO_BU", "VREG"]:
    print(df[col].value_counts(dropna=False), "\n")

PF_PJ
2    1719
1     197
Name: count, dtype: int64 

SEG_NOVO_BU
2    1068
1     514
3     334
Name: count, dtype: int64 

VREG
1    466
2    454
3    325
4    311
6    246
5    114
Name: count, dtype: int64 



In [47]:
df_margins = pd.DataFrame(
    {
        "Colunas": ["PF_PJ", "PF_PJ", "SEG_NOVO_BU", "SEG_NOVO_BU", "SEG_NOVO_BU", "VREG", "VREG", "VREG", "VREG", "VREG", "VREG"],
        "Codigo": [1, 2, 1, 2, 3, 1, 2, 3, 4, 5, 6],
        "Label": ["PF", "PJ", "Empreendedores", "Varejo", "GC", "São Paulo", "Sudeste (Sem São Paulo)", "Sul", "Centro-Oeste", "Norte", "Nordeste"],
        "Total marginais": [214, 1702, 277, 1275, 364, 531, 373, 309, 209, 123, 371]
    }
)
display(df_margins)

# df_margins -> margins (dict de dict)
margins = (
    df_margins.dropna(subset=["Total marginais"])
        .groupby("Colunas")
        .apply(lambda g: dict(zip(g["Codigo"], g["Total marginais"])), include_groups=False)
        .to_dict()
)
print("\n", margins)

,Colunas,Codigo,Label,Total marginais
0,PF_PJ,1,PF,214
1,PF_PJ,2,PJ,1702
2,SEG_NOVO_BU,1,Empreendedores,277
3,SEG_NOVO_BU,2,Varejo,1275
4,SEG_NOVO_BU,3,GC,364
5,VREG,1,São Paulo,531
6,VREG,2,Sudeste (Sem São Paulo),373
7,VREG,3,Sul,309
8,VREG,4,Centro-Oeste,209
9,VREG,5,Norte,123



 {'PF_PJ': {1: 214, 2: 1702}, 'SEG_NOVO_BU': {1: 277, 2: 1275, 3: 364}, 'VREG': {1: 531, 2: 373, 3: 309, 4: 209, 5: 123, 6: 371}}


In [48]:
# Margens populacionais (totais)
# margins = {
#     "sexo": {"M": 5200, "F": 4800},
#     "idade": {"18-34": 3000, "35-54": 4000, "55+": 3000}
# }

margins = {
    "PF_PJ": {1: 214, 2: 1702},
    "SEG_NOVO_BU": {1: 277, 2: 1275, 3: 364},
    "VREG": {1: 531, 2: 373, 3: 309, 4: 209, 5: 123, 6: 371}
}
margins

{'PF_PJ': {1: 214, 2: 1702},
 'SEG_NOVO_BU': {1: 277, 2: 1275, 3: 364},
 'VREG': {1: 531, 2: 373, 3: 309, 4: 209, 5: 123, 6: 371}}

In [49]:
# 1) Rake no peso principal
w_raked = rake_weights(df, "w_base", margins, control={"maxit": 5000, "epsilon": 1e-10})
df["w_raked"] = w_raked
display(df)

print("Checagem das margens (peso principal raked):")
checks = check_margins(df, df["w_raked"].values, margins)
for var, tab in checks.items():
    print("\n", var)
    print(tab)

,ID_EMP,ONDA,TRAB_C,PF_PJ,SEG_NOVO_BU,VREG,POND,w_base,w_raked
0,3000230,59,1,2,2,2,0.999119,1.010489,1.010489
2,3000383,59,1,2,1,2,0.317763,0.380962,0.380962
9,3001301,59,1,1,1,2,0.470412,0.583839,0.583839
10,3000014,59,1,2,2,3,0.994190,1.112214,1.112214
13,3000716,59,1,2,2,1,1.340074,1.347854,1.347854
...,...,...,...,...,...,...,...,...,...
15351,3004051,59,1,2,3,3,1.167923,0.973521,0.973521
15352,3004052,59,1,2,3,4,0.811759,0.676659,0.676659
15353,3004053,59,1,2,3,2,1.183012,0.884481,0.884481
15354,3004054,59,1,2,3,3,1.167923,0.973521,0.973521


Checagem das margens (peso principal raked):

 PF_PJ
   weighted_total  population_total
1           214.0               214
2          1702.0              1702

 SEG_NOVO_BU
   weighted_total  population_total
1           277.0               277
2          1275.0              1275
3           364.0               364

 VREG
   weighted_total  population_total
1           531.0               531
2           373.0               373
3           309.0               309
4           209.0               209
5           123.0               123
6           371.0               371


In [50]:
# 2) Replicate weights (bootstrap) + rake em cada replicate
R = 20
Wrep = make_bootstrap_replicates(df, "w_base", R=R, seed=2026)
print(Wrep_raked)

Wrep_raked = np.zeros_like(Wrep)
for r in range(R):
    df["_wrep"] = Wrep[:, r]
    Wrep_raked[:, r] = rake_weights(df, "_wrep", margins, control={"maxit": 100, "epsilon": 1e-10})
print(Wrep_raked)

NameError: name 'Wrep_raked' is not defined

In [ ]:
# Exemplo: estimar uma média ponderada de uma variável y e erro padrão via replicates
# Vamos criar y fictício
df["y"] = [10, 12, 14, 11, 9, 13, 8, 15, 7, 16]

def wmean(y, w):
    return np.sum(y * w) / np.sum(w)

theta = wmean(df["y"].values, df["w_raked"].values)
thetas_rep = np.array([wmean(df["y"].values, Wrep_raked[:, r]) for r in range(R)])
se_boot = np.sqrt(np.mean((thetas_rep - theta) ** 2))

print("\nEstimativa (média ponderada) com rake:", theta)
print("Erro padrão (bootstrap replicates raked):", se_boot)


Estimativa (média ponderada) com rake: 11.58350482571519
Erro padrão (bootstrap replicates raked): nan


C:\Users\rayner.santos\AppData\Local\Temp\ipykernel_4904\2477881720.py:6: RuntimeWarning: invalid value encountered in scalar divide
  return np.sum(y * w) / np.sum(w)


## <font color="greenblue"> Ponderação RAKE (IPF) - por quotas/percentuais </font>

In [1]:
"""
Rake (IPF) por QUOTAS (percentuais) no estilo da macro SPSS

✅ Você informa:
- N: total final desejado (soma dos pesos após ponderar)
- w_vars: lista de triplas (variável, código, quota) igual ao SPSS
- df: seu DataFrame contendo as colunas dessas variáveis (com códigos)

A função:
- padroniza quotas por variável (se não somarem 1, ela normaliza)
- converte quota -> total-alvo absoluto (N * quota_normalizada)
- faz raking iterativo (alternando variáveis) até convergir ou atingir maxit
- retorna o vetor de pesos e também as tabelas de checagem
"""

from __future__ import annotations
from dataclasses import dataclass
from typing import Dict, Iterable, List, Tuple, Union, Optional
import numpy as np
import pandas as pd


@dataclass
class RakeResult:
    weights: pd.Series
    targets_abs: Dict[str, Dict[Union[int, str], float]]
    iterations: int
    converged: bool
    max_rel_change: float
    checks: Dict[str, pd.DataFrame]


def _parse_w_vars(
    w_vars: Iterable[Tuple[str, Union[int, str], float]]
) -> Dict[str, Dict[Union[int, str], float]]:
    """
    Converte lista de triplas (var, code, quota) em:
    {var: {code: quota, ...}, ...}
    """
    out: Dict[str, Dict[Union[int, str], float]] = {}
    for var, code, quota in w_vars:
        if var not in out:
            out[var] = {}
        if code in out[var]:
            raise ValueError(f"Categoria duplicada: var='{var}', code='{code}'.")
        if quota is None or not np.isfinite(quota):
            raise ValueError(f"Quota inválida para var='{var}', code='{code}': {quota}")
        if quota < 0:
            raise ValueError(f"Quota negativa para var='{var}', code='{code}': {quota}")
        out[var][code] = float(quota)
    if not out:
        raise ValueError("w_vars vazio. Informe ao menos uma variável/categoria/quota.")
    return out


def _normalize_quotas(quotas_by_var: Dict[str, Dict[Union[int, str], float]]) -> Dict[str, Dict[Union[int, str], float]]:
    """
    Normaliza as quotas de cada variável para somarem 1.
    (Replicando o comentário da macro SPSS: se quotas não somarem 1, ela reescala.)
    """
    normed: Dict[str, Dict[Union[int, str], float]] = {}
    for var, qdict in quotas_by_var.items():
        s = sum(qdict.values())
        if s <= 0:
            raise ValueError(f"Soma das quotas da variável '{var}' é <= 0. Não é possível normalizar.")
        normed[var] = {code: q / s for code, q in qdict.items()}
    return normed


def _make_targets_abs(
    N: float,
    quotas_normed: Dict[str, Dict[Union[int, str], float]]
) -> Dict[str, Dict[Union[int, str], float]]:
    """
    Converte quota -> total-alvo absoluto: target = N * quota
    """
    return {var: {code: N * q for code, q in qdict.items()} for var, qdict in quotas_normed.items()}


def rake_by_quota_spss_style(
    df: pd.DataFrame,
    *,
    N: float,
    w_vars: Iterable[Tuple[str, Union[int, str], float]],
    weight_col: str = "weight",
    maxit: int = 200,
    epsilon: float = 1e-10,
    clip: Optional[Tuple[float, float]] = None,
    verbose: bool = False,
) -> RakeResult:
    """
    Replica o comportamento essencial da macro SPSS (rake por quotas):
    - Inicializa peso = 1
    - Normaliza quotas por variável (se não somarem 1)
    - Converte quotas para targets absolutos: N * quota
    - Ajusta iterativamente alternando variáveis
    - (Opcional) aplica corte de pesos (clip) se você quiser evitar extremos

    Parâmetros:
      df: DataFrame com colunas das variáveis (ex.: 'TIPO_NIV', 'TIPO_REG')
      N: total final desejado (soma dos pesos)
      w_vars: lista de triplas (var, code, quota) como no SPSS
      weight_col: nome da coluna de peso a ser criada/atualizada
      maxit: iterações máximas
      epsilon: tolerância de convergência (maior variação relativa do peso)
      clip: (min, max) para truncar pesos se necessário (opcional)
      verbose: imprime progresso

    Retorna:
      RakeResult com pesos, targets e tabelas de checagem.
    """
    if N is None or not np.isfinite(N) or N <= 0:
        raise ValueError(f"N inválido: {N}. Deve ser > 0.")

    quotas_by_var = _parse_w_vars(w_vars)
    quotas_normed = _normalize_quotas(quotas_by_var)
    targets_abs = _make_targets_abs(float(N), quotas_normed)

    # checagens de presença de colunas e categorias
    for var, tdict in targets_abs.items():
        if var not in df.columns:
            raise KeyError(f"Coluna '{var}' não existe no df.")
        # missing
        if df[var].isna().any():
            raise ValueError(f"Há missing (NaN) na coluna '{var}'. Trate antes de ponderar.")
        # categorias alvo precisam existir na amostra (senão divisão por zero)
        present = set(df[var].unique().tolist())
        missing_cats = [c for c in tdict.keys() if c not in present]
        if missing_cats:
            raise ValueError(
                f"Categorias {missing_cats} de '{var}' não aparecem na amostra. "
                "Rake por margens 1D não consegue ajustar quando a categoria está ausente."
            )

    # inicializa pesos = 1 (como a macro)
    w = np.ones(len(df), dtype=float)

    # ordem das variáveis (como a macro percorre cada critério)
    var_order = list(targets_abs.keys())

    converged = False
    max_rel_change = np.inf
    it_done = 0

    for it in range(1, maxit + 1):
        w_old = w.copy()

        for var in var_order:
            # total ponderado atual por categoria
            cur = (
                pd.DataFrame({var: df[var].values, "_w": w})
                .groupby(var)["_w"].sum()
            )

            # aplica fator por categoria: target / current
            for code, target in targets_abs[var].items():
                current = float(cur.loc[code])
                if current <= 0:
                    raise ValueError(
                        f"Total ponderado atual 0 para var='{var}', cat='{code}'. "
                        "Isso não deveria ocorrer se a categoria existe na amostra."
                    )
                factor = target / current
                mask = (df[var].values == code)
                w[mask] *= factor

        # opcional: truncar pesos
        if clip is not None:
            lo, hi = clip
            if lo <= 0 or hi <= 0 or lo > hi:
                raise ValueError(f"clip inválido: {clip}. Use (min>0, max>0, min<=max).")
            w = np.clip(w, lo, hi)

        # convergência: maior mudança relativa do peso
        rel = np.max(np.abs(w - w_old) / (np.abs(w_old) + 1e-12))
        max_rel_change = float(rel)
        it_done = it

        if verbose:
            print(f"it={it:3d}  max_rel_change={max_rel_change:.3e}  sum_w={w.sum():.6f}")

        if max_rel_change < epsilon:
            converged = True
            break

    # resultados e checagens (estilo SPSS "quota vs weighted")
    w_series = pd.Series(w, index=df.index, name=weight_col)

    checks: Dict[str, pd.DataFrame] = {}
    for var in var_order:
        weighted = (
            pd.DataFrame({var: df[var].values, "_w": w})
            .groupby(var)["_w"].sum()
            .reindex(list(targets_abs[var].keys()))
        )
        target = pd.Series(targets_abs[var]).reindex(weighted.index).astype(float)

        # percentuais
        weighted_pct = weighted / w.sum()
        target_pct = target / float(N)  # como target = N * quota, isso volta à quota

        checks[var] = pd.DataFrame({
            "target_abs": target.values,
            "weighted_abs": weighted.values,
            "target_pct": target_pct.values,
            "weighted_pct": weighted_pct.values,
        }, index=weighted.index)

    return RakeResult(
        weights=w_series,
        targets_abs=targets_abs,
        iterations=it_done,
        converged=converged,
        max_rel_change=max_rel_change,
        checks=checks
    )

In [21]:
# Importar o dataframe
file_path = r"C:\Users\rayner.santos\Downloads\Consistencia e Processamento\FTD\BD_FTD_Satisfacao 2025_checar Pesos.xlsx"
data = pd.read_excel(io=file_path, sheet_name="BD_CODIGOS_FTD")
data

,id,TIPO_reg,TIPO_NIV,TIPO,REG_POND,codigo_entrevistado,COD_UNICO,CIDADE,UF,REGIONAL,...,USO_MONIT_A,solicitar_canc_A,desc_cancelamento_A,ID_SUPERVISOR_A,DATA_AGENDA_CHEC_A,HORA_AGENDA_CHEC_A,STATUS_CHEC_A,EMUSO_CHEC_A,DATA_TRANSCRICAO_A,POND
0,177,31,31,3,1,10426,265667.0,DUQUE DE CAXIAS,RJ,SUDESTE,...,NaN,NaN,NaN,115.0,NaN,NaN,NaN,NaN,NaN,3.851416
1,179,33,32,3,3,20248,266312.0,RECIFE,PE,NORDESTE,...,cVXCFnRfJsvKpktKE1nOtg==,NaN,NaN,115.0,NaN,NaN,NaN,NaN,NaN,0.260262
2,180,41,42,4,1,190528,NaN,CIDADE TESTE,RJ,NaN,...,cVXCFnRfJsvKpktKE1nOtg==,NaN,NaN,115.0,NaN,NaN,NaN,NaN,NaN,0.658817
3,184,13,12,1,3,24215,77540.0,ESCADA,PE,NORDESTE,...,61P283Gj8kD2FOn3Md1Dug==,NaN,NaN,115.0,NaN,NaN,NaN,NaN,NaN,0.613430
4,191,11,12,1,1,20823,265505.0,SAO VICENTE,SP,SUDESTE,...,NaN,NaN,NaN,115.0,NaN,NaN,NaN,NaN,NaN,0.815009
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
696,3070,41,42,4,1,21198,26977.0,ITAPERUNA,RJ,SUDESTE,...,NaN,NaN,NaN,115.0,NaN,NaN,NaN,NaN,NaN,0.658817
697,3076,43,42,4,3,17914,28272.0,TERESINA,PI,NORDESTE,...,NaN,NaN,NaN,115.0,NaN,NaN,NaN,NaN,NaN,0.787425
698,3079,43,42,4,3,12414,30522.0,VITORIA DA CONQUISTA,BA,NORDESTE,...,NaN,NaN,NaN,116.0,NaN,NaN,NaN,NaN,NaN,0.787425
699,3081,21,22,2,1,17451,250934.0,CAXAMBU,MG,SUDESTE,...,NaN,NaN,NaN,113.0,NaN,NaN,NaN,NaN,NaN,0.578648


In [26]:
file_path = r"C:\Users\rayner.santos\Downloads\Consistencia e Processamento\FTD\POND_FTD.xlsx"
df_rake = pd.read_excel(file_path, sheet_name="RAKE_QUOTA")
display(df_rake)
w_vars = []
for i, col in enumerate(df_rake["Colunas"]):
    codigo = int(df_rake.loc[i, "Codigo"])
    perc = float(df_rake.loc[i, "Perc marginal"])
    w_vars.append((col, codigo, perc))
print("\n", w_vars)

,Colunas,Codigo,Label,Total marginal,Perc marginal
0,TIPO_NIV,11,FTD SE EI,2731,0.063234
1,TIPO_NIV,12,FTD SE AI,3742,0.086642
2,TIPO_NIV,13,FTD SE AF,1739,0.040265
3,TIPO_NIV,14,FTD SE EM,605,0.014008
4,TIPO_NIV,21,CONC SE EI,5710,0.132210
5,TIPO_NIV,22,CONC SE AI,5829,0.134965
6,TIPO_NIV,23,CONC SE AF,4681,0.108384
7,TIPO_NIV,24,CONC SE EM,2972,0.068814
8,TIPO_NIV,31,FTD DIDAT EI,1911,0.044247
9,TIPO_NIV,32,FTD DIDAT AI,729,0.016879



 [('TIPO_NIV', 11, 0.063233693764616), ('TIPO_NIV', 12, 0.0866424321007664), ('TIPO_NIV', 13, 0.040264882261687), ('TIPO_NIV', 14, 0.0140081965315242), ('TIPO_NIV', 21, 0.132209590404964), ('TIPO_NIV', 22, 0.134964921623562), ('TIPO_NIV', 23, 0.108384079279446), ('TIPO_NIV', 24, 0.0688138183333719), ('TIPO_NIV', 31, 0.0442473778045336), ('TIPO_NIV', 32, 0.0168792979693904), ('TIPO_NIV', 33, 0.0128968024265438), ('TIPO_NIV', 34, 0.00301002570098868), ('TIPO_NIV', 41, 0.0712449929380166), ('TIPO_NIV', 42, 0.0725416193938271), ('TIPO_NIV', 43, 0.0272060015281669), ('TIPO_NIV', 44, 0.00421403598138415), ('TIPO_NIV', 51, 0.00919215540994235), ('TIPO_NIV', 52, 0.0144018152770381), ('TIPO_NIV', 53, 0.0074092940332029), ('TIPO_NIV', 61, 0.0215332607839959), ('TIPO_NIV', 62, 0.0249137511866447), ('TIPO_NIV', 63, 0.0217879552663873), ('TIPO_reg', 11, 0.113416613734208), ('TIPO_reg', 12, 0.0290775636953913), ('TIPO_reg', 13, 0.0781283082786365), ('TIPO_reg', 21, 0.199026042769426), ('TIPO_reg', 

In [27]:
w_vars

[('TIPO_NIV', 11, 0.063233693764616),
 ('TIPO_NIV', 12, 0.0866424321007664),
 ('TIPO_NIV', 13, 0.040264882261687),
 ('TIPO_NIV', 14, 0.0140081965315242),
 ('TIPO_NIV', 21, 0.132209590404964),
 ('TIPO_NIV', 22, 0.134964921623562),
 ('TIPO_NIV', 23, 0.108384079279446),
 ('TIPO_NIV', 24, 0.0688138183333719),
 ('TIPO_NIV', 31, 0.0442473778045336),
 ('TIPO_NIV', 32, 0.0168792979693904),
 ('TIPO_NIV', 33, 0.0128968024265438),
 ('TIPO_NIV', 34, 0.00301002570098868),
 ('TIPO_NIV', 41, 0.0712449929380166),
 ('TIPO_NIV', 42, 0.0725416193938271),
 ('TIPO_NIV', 43, 0.0272060015281669),
 ('TIPO_NIV', 44, 0.00421403598138415),
 ('TIPO_NIV', 51, 0.00919215540994235),
 ('TIPO_NIV', 52, 0.0144018152770381),
 ('TIPO_NIV', 53, 0.0074092940332029),
 ('TIPO_NIV', 61, 0.0215332607839959),
 ('TIPO_NIV', 62, 0.0249137511866447),
 ('TIPO_NIV', 63, 0.0217879552663873),
 ('TIPO_reg', 11, 0.113416613734208),
 ('TIPO_reg', 12, 0.0290775636953913),
 ('TIPO_reg', 13, 0.0781283082786365),
 ('TIPO_reg', 21, 0.19902604

In [31]:
# ----------------------------
# Exemplo de uso 
# ----------------------------

df = data.copy()
# w_vars = [
#     ("TIPO_NIV", 11, 0.063233693764616),
#     ("TIPO_NIV", 12, 0.0866424321007664),
#     ("TIPO_NIV", 13, 0.040264882261687),
#     ("TIPO_NIV", 14, 0.0140081965315242),
#     ("TIPO_NIV", 21, 0.132209590404964),
#     ("TIPO_NIV", 22, 0.134964921623562),
#     ("TIPO_NIV", 23, 0.108384079279446),
#     ("TIPO_NIV", 24, 0.0688138183333719),
#     ("TIPO_NIV", 31, 0.0442473778045336),
#     ("TIPO_NIV", 32, 0.0168792979693904),
#     ("TIPO_NIV", 33, 0.0128968024265438),
#     ("TIPO_NIV", 34, 0.00301002570098868),
#     ("TIPO_NIV", 41, 0.0712449929380166),
#     ("TIPO_NIV", 42, 0.0725416193938271),
#     ("TIPO_NIV", 43, 0.0272060015281669),
#     ("TIPO_NIV", 44, 0.00421403598138415),
#     ("TIPO_NIV", 51, 0.00919215540994235),
#     ("TIPO_NIV", 52, 0.0144018152770381),
#     ("TIPO_NIV", 53, 0.0074092940332029),
#     ("TIPO_NIV", 61, 0.0215332607839959),
#     ("TIPO_NIV", 62, 0.0249137511866447),
#     ("TIPO_NIV", 63, 0.0217879552663873),

#     ("TIPO_reg", 11, 0.113416613734208),
#     ("TIPO_reg", 12, 0.0290775636953913),
#     ("TIPO_reg", 13, 0.0781283082786365),
#     ("TIPO_reg", 21, 0.199026042769426),
#     ("TIPO_reg", 22, 0.0445338414849319),
#     ("TIPO_reg", 23, 0.109746629966829),
#     ("TIPO_reg", 31, 0.0668360505328534),
#     ("TIPO_reg", 32, 0.018138189004164),
#     ("TIPO_reg", 33, 0.0468628696449996),
#     ("TIPO_reg", 41, 0.067612393252876),
#     ("TIPO_reg", 42, 0.0108687980803162),
#     ("TIPO_reg", 43, 0.117439480556144),
#     ("TIPO_reg", 51, 0.0214552897169878),
#     ("TIPO_reg", 52, 0.00663420142564754),
#     ("TIPO_reg", 53, 0.008328040087515),
#     ("TIPO_reg", 61, 0.0417107770484861),
#     ("TIPO_reg", 62, 0.0115039875785165),
#     ("TIPO_reg", 63, 0.00868092314207072),
# ]

res = rake_by_quota_spss_style(
    df,
    N=701,
    w_vars=w_vars,
    weight_col="weight",
    maxit=200,
    epsilon=1e-10,
    clip=None,
    verbose=True
)

df["POND_RAKE"] = res.weights
print("\nConvergiu?", res.converged, "Iterações:", res.iterations, "Max rel change:", res.max_rel_change)
print("\nChecagem TIPO_NIV (primeiras linhas):")
print(res.checks["TIPO_NIV"].head())
print("\nChecagem TIPO_reg (primeiras linhas):")
print(res.checks["TIPO_reg"].head())
print("\nSoma dos pesos:", df["POND_RAKE"].sum())

it=  1  max_rel_change=3.098e+00  sum_w=701.000000
it=  2  max_rel_change=1.667e-01  sum_w=701.000000
it=  3  max_rel_change=1.085e-02  sum_w=701.000000
it=  4  max_rel_change=1.403e-03  sum_w=701.000000
it=  5  max_rel_change=1.832e-04  sum_w=701.000000
it=  6  max_rel_change=2.923e-05  sum_w=701.000000
it=  7  max_rel_change=4.766e-06  sum_w=701.000000
it=  8  max_rel_change=7.771e-07  sum_w=701.000000
it=  9  max_rel_change=1.267e-07  sum_w=701.000000
it= 10  max_rel_change=2.066e-08  sum_w=701.000000
it= 11  max_rel_change=3.368e-09  sum_w=701.000000
it= 12  max_rel_change=5.491e-10  sum_w=701.000000
it= 13  max_rel_change=8.953e-11  sum_w=701.000000

Convergiu? True Iterações: 13 Max rel change: 8.953215071376327e-11

Checagem TIPO_NIV (primeiras linhas):
          target_abs  weighted_abs  target_pct  weighted_pct
TIPO_NIV                                                    
11         44.326819     47.903655    0.063234      0.068336
12         60.736345     65.637304    0.086642

In [32]:
df

,id,TIPO_reg,TIPO_NIV,TIPO,REG_POND,codigo_entrevistado,COD_UNICO,CIDADE,UF,REGIONAL,...,solicitar_canc_A,desc_cancelamento_A,ID_SUPERVISOR_A,DATA_AGENDA_CHEC_A,HORA_AGENDA_CHEC_A,STATUS_CHEC_A,EMUSO_CHEC_A,DATA_TRANSCRICAO_A,POND,POND_RAKE
0,177,31,31,3,1,10426,265667.0,DUQUE DE CAXIAS,RJ,SUDESTE,...,NaN,NaN,115.0,NaN,NaN,NaN,NaN,NaN,3.851416,3.851416
1,179,33,32,3,3,20248,266312.0,RECIFE,PE,NORDESTE,...,NaN,NaN,115.0,NaN,NaN,NaN,NaN,NaN,0.260262,0.260262
2,180,41,42,4,1,190528,NaN,CIDADE TESTE,RJ,NaN,...,NaN,NaN,115.0,NaN,NaN,NaN,NaN,NaN,0.658817,0.658817
3,184,13,12,1,3,24215,77540.0,ESCADA,PE,NORDESTE,...,NaN,NaN,115.0,NaN,NaN,NaN,NaN,NaN,0.613430,0.613430
4,191,11,12,1,1,20823,265505.0,SAO VICENTE,SP,SUDESTE,...,NaN,NaN,115.0,NaN,NaN,NaN,NaN,NaN,0.815009,0.815009
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
696,3070,41,42,4,1,21198,26977.0,ITAPERUNA,RJ,SUDESTE,...,NaN,NaN,115.0,NaN,NaN,NaN,NaN,NaN,0.658817,0.658817
697,3076,43,42,4,3,17914,28272.0,TERESINA,PI,NORDESTE,...,NaN,NaN,115.0,NaN,NaN,NaN,NaN,NaN,0.787425,0.787425
698,3079,43,42,4,3,12414,30522.0,VITORIA DA CONQUISTA,BA,NORDESTE,...,NaN,NaN,116.0,NaN,NaN,NaN,NaN,NaN,0.787425,0.787425
699,3081,21,22,2,1,17451,250934.0,CAXAMBU,MG,SUDESTE,...,NaN,NaN,113.0,NaN,NaN,NaN,NaN,NaN,0.578648,0.578648


In [53]:
df_teste = pd.DataFrame(
    {
        "REG":   ["SUDESTE",            "SUDESTE",           "SUDESTE",    "SUL",        "SUL",                "NORDESTE"],
        "NIVEL": ["Fun. Anos Iniciais", "Fund. Anos Finais", "Ens. Médio", "Ens. Médio", "Fun. Anos Iniciais", "Ens. Médio"],
        "POND":  [ 0.9,                  1.1,                 0.6,          0.5,          1.5,                  1.6]
    }
)
# >>> soma da ponderação por código (inclui NaN do código se houver)
freq = (
    df_teste[["REG", "POND"]]
    .groupby("REG", dropna=False)["POND"]
    .sum()
    .rename("Frequência Ponderada")
    .to_frame()
)

# % sobre o total ponderado
freq["%"] = freq["Frequência Ponderada"] / freq["Frequência Ponderada"].sum() * 100

# linha total
total_line = pd.DataFrame(
    {"Frequência Ponderada": [freq["Frequência Ponderada"].sum()], "%": [100.0]},
    index=["Total"]
)
freq = pd.concat([freq, total_line], axis=0)
freq

,Frequência Ponderada,%
NORDESTE,1.6,25.806452
SUDESTE,2.6,41.935484
SUL,2.0,32.258065
Total,6.2,100.000000


In [34]:
freq = df["TIPO_reg"].value_counts(dropna=False).rename("Frequência").to_frame()
freq

,Frequência
TIPO_reg,
21,121
23,112
13,72
11,71
43,70
41,47
33,45
31,40
22,32
